# 01 — Data Overview

This notebook explores the final monthly dataset used as input for all modeling notebooks.
The dataset contains **309 months (2000-03 to 2025-11)** and **48 features** spanning five groups:
constructed factor returns, Fama-French benchmarks, macro/market-state variables, WSJ news sentiment,
and Reddit/WallStreetBets sentiment.

**Sections:**
1. Load & Inspect
2. Feature Groups
3. Summary Statistics
4. Correlation Heatmap
5. Time-Series Plots

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 120)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

## 1. Load & Inspect

In [ ]:
df = pd.read_csv('data/FinalMonthlyDataset_ours_ff_macro.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Months: {df["date"].nunique()}')
df.head()

In [ ]:
# Missing value overview
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
missing_df[missing_df['missing_count'] > 0].sort_values('missing_%', ascending=False)

## 2. Feature Groups

The 48 columns fall into five categories. Understanding the feature groups is important
before modeling: factor return prediction uses macro + sentiment features as *predictors*,
while the factor returns themselves are the *targets*.

In [ ]:
feature_groups = {
    'Factor Returns (gross)': ['fac_value', 'fac_momentum', 'fac_quality', 'fac_investment', 'fac_size'],
    'Factor Returns (net of TC)': ['fac_value_net', 'fac_momentum_net', 'fac_quality_net', 'fac_investment_net', 'fac_size_net'],
    'Portfolio Metadata': [c for c in df.columns if c.startswith(('n_long', 'n_short', 'to_'))],
    'Fama-French Benchmarks': ['hml', 'umd', 'smb', 'rmw', 'cma', 'mktrf', 'rf'],
    'Macro / Market State': ['fedfunds', 'dgs10', 'term_spread_10y_fedfunds', 'mkt_vol_12m',
                             'mkt_trend_12m', 'dispersion', 'rate_chg_3m', 'rate_chg_12m'],
    'WSJ News Sentiment': ['wsj_index', 'wsj_uncertainty', 'topic_index', 'topic_uncertainty'],
    'Reddit / WallStreetBets': ['WallStreetBets_score', 'WallStreetBets_confidence', 'WallStreetBets_numeric_score'],
}

rows = []
for group, cols in feature_groups.items():
    available = [c for c in cols if c in df.columns]
    rows.append({'Group': group, 'Count': len(available), 'Features': ', '.join(available)})
pd.DataFrame(rows).set_index('Group')

## 3. Summary Statistics

Key statistics for the modeling features. Note the different data availability windows:
- **Factor returns**: full sample (2000-03 onwards), with a brief warm-up period
- **WSJ sentiment**: 2000-03 → 2017-06 (208 months)
- **Reddit sentiment**: 2013-01 → 2025-11 (156 months)

In [ ]:
key_cols = [
    'fac_value_net', 'fac_momentum_net', 'fac_quality_net', 'fac_investment_net', 'fac_size_net',
    'mktrf', 'term_spread_10y_fedfunds', 'mkt_vol_12m', 'dispersion',
    'wsj_index', 'wsj_uncertainty', 'topic_index',
    'WallStreetBets_score', 'WallStreetBets_confidence',
]
key_cols = [c for c in key_cols if c in df.columns]

stats = df[key_cols].describe().T
stats['ann_sharpe'] = np.sqrt(12) * stats['mean'] / stats['std']
stats[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'ann_sharpe']].round(4)

In [ ]:
# Factor return distributions
factor_net_cols = ['fac_value_net', 'fac_momentum_net', 'fac_quality_net', 'fac_investment_net', 'fac_size_net']
factor_labels = ['Value', 'Momentum', 'Quality', 'Investment', 'Size']
factor_net_cols = [c for c in factor_net_cols if c in df.columns]

fig, axes = plt.subplots(1, len(factor_net_cols), figsize=(15, 3), sharey=False)
colors = ['#2196F3', '#E91E63', '#4CAF50', '#FF9800', '#9C27B0']

for ax, col, lbl, color in zip(axes, factor_net_cols, factor_labels, colors):
    data = df[col].dropna()
    ax.hist(data, bins=30, color=color, alpha=0.75, edgecolor='white', linewidth=0.5)
    ax.axvline(data.mean(), color='black', linestyle='--', linewidth=1.2, label=f'Mean: {data.mean():.3f}')
    ax.set_title(lbl, fontweight='bold')
    ax.set_xlabel('Monthly Return')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle('Net Factor Return Distributions (monthly)', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Correlation Heatmap

Correlations across key modeling features. The factor returns should show low mutual correlation
(otherwise diversification across factors would be less valuable). Sentiment and macro features
should ideally show low correlation with each other to provide complementary information.

In [ ]:
corr_cols = [
    'fac_value_net', 'fac_momentum_net', 'fac_quality_net', 'fac_investment_net', 'fac_size_net',
    'mktrf', 'term_spread_10y_fedfunds', 'mkt_vol_12m', 'mkt_trend_12m', 'dispersion',
    'wsj_index', 'wsj_uncertainty', 'WallStreetBets_score',
]
corr_cols = [c for c in corr_cols if c in df.columns]
corr_labels = {
    'fac_value_net': 'Value', 'fac_momentum_net': 'Momentum', 'fac_quality_net': 'Quality',
    'fac_investment_net': 'Investment', 'fac_size_net': 'Size',
    'mktrf': 'Mkt-RF', 'term_spread_10y_fedfunds': 'Term Spread',
    'mkt_vol_12m': 'Mkt Vol', 'mkt_trend_12m': 'Mkt Trend', 'dispersion': 'Dispersion',
    'wsj_index': 'WSJ Index', 'wsj_uncertainty': 'WSJ Uncert.', 'WallStreetBets_score': 'WSB Score',
}

corr_matrix = df[corr_cols].corr()
display_labels = [corr_labels.get(c, c) for c in corr_cols]

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8)

ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(display_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(display_labels, fontsize=9)

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        val = corr_matrix.values[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=7, color='black' if abs(val) < 0.6 else 'white')

ax.set_title('Pairwise Correlations — Key Features', fontsize=12, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

## 5. Time-Series Plots

Visualizing the features over time reveals structural shifts and regime changes that the ML models
will need to navigate — e.g., the 2008 financial crisis, the 2020 COVID shock, and the 2022
rate-hiking cycle.

In [ ]:
# Cumulative factor returns over time
df_plot = df.set_index('date')

fig, ax = plt.subplots(figsize=(13, 5))
colors = ['#2196F3', '#E91E63', '#4CAF50', '#FF9800', '#9C27B0']

for col, lbl, color in zip(factor_net_cols, factor_labels, colors):
    data = df_plot[col].dropna()
    cum = (1 + data).cumprod()
    ax.plot(cum.index, cum.values, label=lbl, color=color, linewidth=1.5)

ax.axhline(1.0, color='grey', linewidth=0.7, linestyle=':')
ax.set_title('Cumulative Net Factor Returns (2000–2025)', fontsize=12, fontweight='bold')
ax.set_ylabel('Growth of $1')
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Macro and sentiment features over time
macro_plot_cols = [
    ('term_spread_10y_fedfunds', 'Term Spread (10Y - Fed Funds)', '#1565C0'),
    ('mkt_vol_12m',              '12M Market Volatility',         '#C62828'),
    ('wsj_index',                'WSJ Sentiment Index',           '#2E7D32'),
    ('WallStreetBets_score',     'Reddit/WSB Sentiment Score',    '#6A1B9A'),
]
macro_plot_cols = [(c, l, col) for c, l, col in macro_plot_cols if c in df_plot.columns]

fig, axes = plt.subplots(len(macro_plot_cols), 1, figsize=(13, 8), sharex=True)

for ax, (col, label, color) in zip(axes, macro_plot_cols):
    data = df_plot[col].dropna()
    ax.plot(data.index, data.values, color=color, linewidth=1.2)
    ax.axhline(data.mean(), color='black', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.set_ylabel(label, fontsize=9)
    ax.grid(alpha=0.2)
    ax.fill_between(data.index, data.values, data.mean(),
                    where=(data.values > data.mean()),
                    alpha=0.15, color=color)

fig.suptitle('Key Predictor Variables Over Time', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()